# Common Settings

In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Looking Inside Large Language Models

In [2]:
# import torch <-- comment it which may cause kernel corrupt when there is no enough memory
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    #device_map="cuda",
    device_map="mps",  # Use MPS for Apple Silicon instead of "cuda"
    torch_dtype="auto",
    #trust_remote_code=True,
    local_files_only=True,
    attn_implementation="eager"
)

# Create a pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=50,
    do_sample=False,
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

# The Inputs and Outputs of a Trained Transformer LLM

In [3]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

output = generator(prompt)

print(output[0]['generated_text'])

You are not running the flash-attention implementation, expect numerical differences.


 Mention the steps you're taking to prevent it in the future.

Dear Sarah,

I hope this message finds you well. I am writing to express my sincerest apologies for the unfortunate incident that occurred


In [4]:
print(model)

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
          (rotary_emb): Phi3RotaryEmbedding()
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLU()
        )
        (input_layernorm): Phi3RMSNorm()
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
        (post_attention_layernorm): Phi3RMSNorm()
      )
    )
    (norm): Phi3RMSNorm()
  )
  (lm_head): Linear(in_features=3072, out_features=3206

# Choosing a Single Token from the Probability Distribution (Sampling/Decoding)

In [21]:
prompt = "The capital of France is"

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
print(input_ids)

# Tokenize the input prompt
#input_ids = input_ids.to("cuda")
input_ids = input_ids.to("mps")
print(input_ids)

# Get the output of the model before the lm_head
model_output = model.model(input_ids)
#print(model_output)

# Get the output of the lm_head
lm_head_output = model.lm_head(model_output[0])
print(lm_head_output)
print(lm_head_output[0,0])

tensor([[ 450, 7483,  310, 3444,  338]])
tensor([[ 450, 7483,  310, 3444,  338]], device='mps:0')
tensor([[[25.0000, 25.1250, 23.0000,  ..., 19.1250, 19.1250, 19.1250],
         [31.0000, 31.5000, 26.0000,  ..., 25.8750, 25.8750, 25.8750],
         [31.5000, 28.8750, 31.0000,  ..., 26.2500, 26.2500, 26.2500],
         [33.0000, 31.8750, 36.0000,  ..., 27.8750, 27.8750, 27.8750],
         [27.8750, 29.5000, 28.2500,  ..., 20.5000, 20.5000, 20.5000]]],
       device='mps:0', dtype=torch.bfloat16, grad_fn=<LinearBackward0>)
tensor([25.0000, 25.1250, 23.0000,  ..., 19.1250, 19.1250, 19.1250],
       device='mps:0', dtype=torch.bfloat16, grad_fn=<SelectBackward0>)


'Paris'

In [22]:
token_id = lm_head_output[0,-1].argmax(-1)
tokenizer.decode(token_id)

'Paris'

In [23]:
model_output[0].shape

torch.Size([1, 5, 3072])

In [24]:
lm_head_output.shape

torch.Size([1, 5, 32064])